## Import necessary packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import anndata as ad
import scanpy as sc

from CellPLM.utils import set_seed
from CellPLM.pipeline.cell_type_annotation import CellTypeAnnotationPipeline, CellTypeAnnotationDefaultPipelineConfig, CellTypeAnnotationDefaultModelConfig

In [ ]:
# Set a seed for reproducibility

import os
import torch
import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

## Specify important parameters before getting started

In [ ]:
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'

## Load Downstream Dataset

In [ ]:
set_seed(42)

In [ ]:
data = ad.read_h5ad(f'../data/gse155468.h5ad')

In [ ]:
# keep raw counts
data.layers["counts"] = data.X.copy()

# normalize each cell to target sum (e.g., 1e4 = CP10k)
sc.pp.normalize_total(data, target_sum=1e4)   # in-place on adata.X

# log1p transform (natural log)
sc.pp.log1p(data)

In [ ]:
from sklearn.model_selection import train_test_split

TARGET_COL = 'celltype'

mask_labeled = data.obs[TARGET_COL].notna() & (data.obs[TARGET_COL].astype(str).str.len() > 0)
n_dropped = (~mask_labeled).sum()
if n_dropped:
    print(f"Dropping {n_dropped} unlabeled cells")
data = data[mask_labeled].copy()

y = data.obs[TARGET_COL].astype(str)


train_valid_indices, test_indices = train_test_split(np.arange(data.n_obs), test_size=0.2, random_state=seed, stratify=y)

y_train_valid = y.iloc[train_valid_indices]

train_indices, validation_indices = train_test_split(train_valid_indices, test_size=1/8, random_state=seed, stratify = y_train_valid)

data.obs['split'] = 'train'
validation_labels = data.obs.index[validation_indices]
data.obs.loc[validation_labels, 'split'] = 'valid'
test_labels = data.obs.index[test_indices]
data.obs.loc[test_labels, 'split'] = 'test'

split_counts = data.obs['split'].value_counts()

print(f"Number of 'train' observations: {split_counts.get('train', 0)}")
print(f"Number of 'valid' observations: {split_counts.get('valid', 0)}")
print(f"Number of 'test' observations: {split_counts.get('test', 0)}")

print(f"Total number of observations: {data.n_obs}")

## Overwrite parts of the default config
These hyperparameters are recommended for general purpose. We did not tune it for individual datasets. You may update them if needed.

In [ ]:
pipeline_config = CellTypeAnnotationDefaultPipelineConfig.copy()

model_config = CellTypeAnnotationDefaultModelConfig.copy()
model_config['out_dim'] = data.obs['celltype'].nunique()
pipeline_config, model_config

## Fine-tuning

Efficient data setup and fine-tuning can be seamlessly conducted using the CellPLM built-in `pipeline` module.

First, initialize a `CellTypeAnnotationPipeline`. This pipeline will automatically load a pretrained model.

In [ ]:
pipeline = CellTypeAnnotationPipeline(pretrain_prefix=PRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                      overwrite_config=model_config,  # This is for overwriting part of the pretrain config
                                      pretrain_directory='../ckpt')
pipeline.model

Next, employ the `fit` function to fine-tune the model on your downstream dataset. This dataset should be in the form of an AnnData object, where `.X` is a csr_matrix, and `.obs` includes information for train-test splitting and cell type labels.

Typically, a dataset containing approximately 20,000 cells can be trained in under 10 minutes using a V100 GPU card, with an expected GPU memory consumption of around 8GB.

In [ ]:
pipeline.fit(data, # An AnnData object
            pipeline_config, # The config dictionary we created previously, optional
            split_field = 'split', #  Specify a column in .obs that contains split information
            train_split = 'train',
            valid_split = 'valid',
            label_fields = ['celltype']) # Specify a column in .obs that contains cell type labels

## Inference and evaluation
Once the pipeline has been fitted to the downstream datasets, performing inference or evaluation on new datasets can be easily accomplished using the built-in `predict` and `score` functions.

In [ ]:
predictions, proba, metrics = pipeline.score(data, # An AnnData object
                pipeline_config, # The config dictionary we created previously, optional
                split_field = 'split', # Specify a column in .obs to specify train and valid split, optional
                target_split = 'test', # Specify a target split to predict, optional
                label_fields = ['celltype'])  # Specify a column in .obs that contains cell type labels

In [ ]:
metrics

In [ ]:
encoder = pipeline.label_encoders['celltype']
pred_labels = encoder.inverse_transform(predictions.cpu().numpy().astype(int))

In [ ]:
data_test = data[data.obs['split'] == 'test'].copy()

data_test.obs['predictions'] = pred_labels

In [ ]:
# Calculate PR-AUC and ROC-AUC metrics

import numpy as np
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, roc_auc_score

def macro_pr_roc_auc(y_true, proba):
    """
    y_true: (N,) integer class ids in [0, K-1]
    proba : (N, K) per-class probabilities (rows sum ~ 1)
    Returns (pr_auc_macro, roc_auc_macro)
    """
    y_true = np.asarray(y_true)
    proba  = np.asarray(proba)
    K = proba.shape[1]

    if y_true.min() < 0 or y_true.max() >= K:
        raise ValueError("y_true must be integer-coded in [0, K-1] matching proba columns.")
    if proba.ndim != 2 or proba.shape[0] != y_true.shape[0]:
        raise ValueError("proba must be (N, K) aligned with y_true (N,)")

    y_true_bin = label_binarize(y_true, classes=np.arange(K))  # (N, K)

    pr_auc_macro  = average_precision_score(y_true_bin, proba, average="macro")
    roc_auc_macro = roc_auc_score(y_true, proba, average="macro", multi_class="ovr")
    return {"pr_auc_macro": pr_auc_macro, "roc_auc_macro": roc_auc_macro}

labels_test = data.obs.loc[data.obs['split'] == 'test', 'celltype']

y_true = encoder.transform(labels_test)           # strings → ints aligned to proba

auc_metrics = macro_pr_roc_auc(y_true, proba[0])


In [ ]:
auc_metrics

In [ ]:
data_test = data[data.obs['split'] == 'test'].copy()

data_test.obs['predictions'] = pred_labels

## Save results to npz file

In [ ]:
Model_name='cellplm'
step='cell_type_annotation'
dataset='ATAA'

In [ ]:
import numpy as np

ACC, PRE, REC, F1 = float(metrics['acc']), float(metrics['precision']), float(metrics['recall']), float(metrics['f1_score'])
PR_AUC, ROC_AUC = float(auc_metrics['pr_auc_macro']), float(auc_metrics['roc_auc_macro'])

np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_{dataset}.npz",
    predictions=data_test.obs['predictions'],
    labels=data_test.obs['celltype'],
    ACC=ACC, PRE=PRE, REC=REC, F1=F1, 
    PR_AUC=PR_AUC, ROC_AUC=ROC_AUC 
)